# 22｜生成模型_自編碼器GAN與擴散

用線性欠完備自編碼器的等價案例 PCA 觀察壓縮與重建誤差。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
digits = load_digits()
X = digits.data / 16.0
pca = PCA(n_components=12, random_state=SEED).fit(X)
Z = pca.transform(X); recon = pca.inverse_transform(Z)
errors = np.mean((X-recon)**2, axis=1)
idx = [0, 1, int(np.argmax(errors))]
pd.DataFrame({'sample': idx, 'label': digits.target[idx], 'reconstruction_MSE': errors[idx]}).round(4)

## 執行結果圖

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(7, 5))
for col, i in enumerate(idx):
    axes[0,col].imshow(X[i].reshape(8,8), cmap='gray'); axes[0,col].set_title(f'original {digits.target[i]}')
    axes[1,col].imshow(recon[i].reshape(8,8), cmap='gray'); axes[1,col].set_title(f'reconstruction\nMSE={errors[i]:.3f}')
    axes[0,col].axis('off'); axes[1,col].axis('off')
savefig('22_generative_demo')
report('generative_models', {'components': 12, 'mean_MSE': errors.mean(), 'samples': idx})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。